## 1. Load dataset (DINOv2 – neutral)

This notebook evaluates the predictive performance of DINOv2 embeddings on the neutral dataset.

In [1]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler

# Load dataset
df = pd.read_csv("../Embeddings/dinov2/final_dataset_neutral_with_dinov2.csv")

print("Dataset shape:", df.shape)
df.head(2)

Dataset shape: (6105, 791)


,cat_no,titulo,autor,escuela_obra,tipo_objeto,datacion,tema,is_religious,is_fauna,century,...,emb_758,emb_759,emb_760,emb_761,emb_762,emb_763,emb_764,emb_765,emb_766,emb_767
0,P000002,El juicio de Paris,"Albani, Francesco",Italiana,Pintura,1650 - 1660,NaN,0,1,17th c.,...,-1.047690,-1.745325,3.675401,2.355326,-0.382265,3.596227,1.518095,-0.291360,-2.527206,-1.529001
1,P000006,Sagrada Familia y el cardenal Fernando de Medici,"Allori, Alessandro",Italiana,Pintura,1584,NaN,1,1,16th c.,...,-0.586006,0.992511,1.241005,1.684799,1.214453,5.411584,1.288250,-1.936812,-1.874689,-0.640553


## 2. Feature matrix and targets

We separate the embedding features (X) from the target variables:
- Warmth
- Competence

In [2]:
# Select embedding columns
embedding_cols = [col for col in df.columns if col.startswith("emb_")]

X = df[embedding_cols].values

# Targets
y_warmth = df["dirmean_Warmth"].values
y_competence = df["dirmean_Competence"].values

print("Feature matrix shape:", X.shape)
print("Number of embedding features:", len(embedding_cols))

Feature matrix shape: (6105, 768)
Number of embedding features: 768


## 3. Train-test split

The dataset is divided into training and test sets using a fixed random seed, so the results remain reproducible and comparable with previous experiments.

In [3]:
X_train, X_test, y_w_train, y_w_test = train_test_split(
    X, y_warmth, test_size=0.2, random_state=42
)

_, _, y_c_train, y_c_test = train_test_split(
    X, y_competence, test_size=0.2, random_state=42
)

print("Train size:", X_train.shape[0])
print("Test size:", X_test.shape[0])

Train size: 4884
Test size: 1221


## 4. Evaluation function

We define a helper function to compute:
- R² (model explanatory power)
- RMSE (prediction error)

In [4]:
def evaluate(y_true, y_pred, name):
    r2 = r2_score(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    
    print(f"{name} R²: {r2:.4f}")
    print(f"{name} RMSE: {rmse:.4f}")
    print("-" * 30)
    
    return r2, rmse

## 5. Mean baseline

We compare all models against a simple baseline that predicts the mean value of the training set.

This helps determine whether the models are actually learning meaningful patterns.

In [5]:
# --- Warmth baseline ---
y_w_pred_baseline = np.full_like(y_w_test, y_w_train.mean())
r2_w_base, rmse_w_base = evaluate(y_w_test, y_w_pred_baseline, "Warmth baseline")

# --- Competence baseline ---
y_c_pred_baseline = np.full_like(y_c_test, y_c_train.mean())
r2_c_base, rmse_c_base = evaluate(y_c_test, y_c_pred_baseline, "Competence baseline")

Warmth baseline R²: -0.0006
Warmth baseline RMSE: 0.5381
------------------------------
Competence baseline R²: -0.0011
Competence baseline RMSE: 0.3184
------------------------------


## 6. Linear model: Ridge regression

Ridge regression tests whether warmth and competence can be recovered through a linear mapping from the embedding space.

In [6]:
# Scale features
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Train models
ridge_w = Ridge(alpha=1.0)
ridge_c = Ridge(alpha=1.0)

ridge_w.fit(X_train_scaled, y_w_train)
ridge_c.fit(X_train_scaled, y_c_train)

# Predictions
y_w_pred_ridge = ridge_w.predict(X_test_scaled)
y_c_pred_ridge = ridge_c.predict(X_test_scaled)

# Evaluate
r2_w_ridge, rmse_w_ridge = evaluate(y_w_test, y_w_pred_ridge, "Warmth Ridge")
r2_c_ridge, rmse_c_ridge = evaluate(y_c_test, y_c_pred_ridge, "Competence Ridge")

Warmth Ridge R²: -0.1315
Warmth Ridge RMSE: 0.5722
------------------------------
Competence Ridge R²: -0.1385
Competence Ridge RMSE: 0.3395
------------------------------


## 7. Non-linear model: Random Forest

A Random Forest model is used to capture non-linear relationships between embeddings and the targets. This helps determine whether the signal exists but is not linearly separable.

In [7]:
rf_w = RandomForestRegressor(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

rf_c = RandomForestRegressor(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

# Train
rf_w.fit(X_train, y_w_train)
rf_c.fit(X_train, y_c_train)

# Predict
y_w_pred_rf = rf_w.predict(X_test)
y_c_pred_rf = rf_c.predict(X_test)

# Evaluate
r2_w_rf, rmse_w_rf = evaluate(y_w_test, y_w_pred_rf, "Warmth RF")
r2_c_rf, rmse_c_rf = evaluate(y_c_test, y_c_pred_rf, "Competence RF")

Warmth RF R²: 0.0158
Warmth RF RMSE: 0.5336
------------------------------
Competence RF R²: -0.0120
Competence RF RMSE: 0.3201
------------------------------


## 8. Results summary

The table below compares baseline, linear, and non-linear models for both targets.

In [8]:
results = pd.DataFrame([
    ["Warmth", "Baseline", r2_w_base, rmse_w_base],
    ["Warmth", "Ridge", r2_w_ridge, rmse_w_ridge],
    ["Warmth", "Random Forest", r2_w_rf, rmse_w_rf],
    
    ["Competence", "Baseline", r2_c_base, rmse_c_base],
    ["Competence", "Ridge", r2_c_ridge, rmse_c_ridge],
    ["Competence", "Random Forest", r2_c_rf, rmse_c_rf]
], columns=["Target", "Model", "R²", "RMSE"])

results

,Target,Model,R²,RMSE
0,Warmth,Baseline,-0.000552,0.538050
1,Warmth,Ridge,-0.131522,0.572182
2,Warmth,Random Forest,0.015758,0.533647
3,Competence,Baseline,-0.001064,0.318350
4,Competence,Ridge,-0.138459,0.339495
5,Competence,Random Forest,-0.012011,0.320086


## Interpretation (DINOv2 – neutral dataset)

The results for DINOv2 on the neutral dataset show very limited predictive performance across all models.

For warmth, the Random Forest achieves a very small positive R² (~0.016), slightly improving over the baseline. However, the gain is minimal, indicating that only a weak signal is present in the embeddings. Ridge regression performs worse than the baseline, suggesting that any relationship between embeddings and warmth is not linear.

For competence, all models fail to outperform the baseline. Both Ridge and Random Forest produce negative R² values, confirming that competence is not meaningfully predictable from visual features in this setup.

Comparing these results with previous experiments, two patterns become clear. First, performance is consistently worse than in the strict dataset, even though the neutral dataset is larger. This shows that adding noisier or imputed labels does not help, and can actually hurt performance. Second, DINOv2 continues to slightly underperform compared to SigLIP, reinforcing the idea that multimodal embeddings capture more relevant semantic information.

Overall, these results confirm that:
- The predictive signal in visual embeddings is very weak  
- Non-linear models help slightly, but not enough  
- Increasing dataset size without improving label quality is not effective  
- Purely visual models (DINOv2) struggle more than multimodal ones (SigLIP)